In [ ]:

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer, clone_chat_template  # setup_chat_format is from trl, not transformers
import torch

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"  # Force CPU for testing purposes



c:\DATEN\educs\Modul 6 KI-Anwendungen\modul_6_KI_anwendung\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── 1. DATASET ──────────────────────────────────────────────
# Use smol-smoltalk (optimized for <1B models, not full smoltalk)
# split= notation returns a Dataset directly (not DatasetDict),
# downloading only 10% of the ~1M rows to save memory/bandwidth
dataset = load_dataset("HuggingFaceTB/smoltalk", "all", split="train[:10%]")

# Create train/eval split (no "test" split exists in this dataset)
split = dataset.train_test_split(test_size=0.05, seed=42)



In [3]:
# ── 2. MODEL & TOKENIZER ────────────────────────────────────
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(model_name)



Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5515.83it/s]


In [4]:
# ── 3. CHAT TEMPLATE ────────────────────────────────────────
# adds special tokens, resizes embeddings, syncs eos_token
model, tokenizer, added_tokens = clone_chat_template(
    model,
    tokenizer,
    source_tokenizer_path="HuggingFaceTB/SmolLM2-135M-Instruct"
)

# clone_chat_template only syncs eos_token — pad_token must be set manually
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token




In [5]:
# ── 4. TRAINING CONFIG ──────────────────────────────────────
# Key parameters and their trade-offs:
#   max_length:                  truncates sequences to fit model context window
#   per_device_train_batch_size: larger = more stable gradients, more VRAM
#   gradient_accumulation_steps: simulates larger batches without extra VRAM
#   learning_rate:               5e-5 is a safe starting point for SFT
#   warmup_ratio:                gradual LR ramp-up avoids early instability
#   eval_strategy + eval_steps:  watch for overfitting (val loss rising while
#                                train loss falls)

training_args = SFTConfig(
    output_dir="./sft_output",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    max_length =2048,        # ✅ truncate at tokenization time, before hitting the model
)



ValueError: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU.

In [ ]:
# ── 5. TRAINER ──────────────────────────────────────────────
# When dataset has a "messages" field, SFTTrainer automatically applies
# the model's chat template — no formatting_func needed
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=tokenizer,
)

# Start training
trainer.train()